## But

Vérifier la validation du pipeline prévu dans AR1

## Principe général

* Historique complet **1959–2025** sur **UNRATE**.
* Données consommées exclusivement via **Feast**.
* Modèle Baseline et univarié.
* Backtesting temporel minimal avec prévisions ponctuelles et intervalles de prédiction conformes (95 %).
* Comparer AR1 et ARp

## Résultat
cf la fin

# Package

In [60]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# ----------------------------
# Nixtla
# ----------------------------
from statsforecast import StatsForecast
from statsforecast.models import AutoRegressive
from statsforecast.utils import ConformalIntervals
from utilsforecast.plotting import plot_series

# Importation des données

In [61]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()


# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Dictionnaire de modèle

In [62]:
# ----------------------------
# (NEW) Dictionnaire modèle basé sur p*
# ----------------------------
SF_MODELS = {
    "AR_pstar": lambda: AutoRegressive(lags=p_star)
}

# Backtesting

In [63]:
def backtest_dynamic_p_oos(
    ts: pd.DataFrame,
    *,
    freq: str,
    h: int,
    step_size: int,
    test_start_ts: pd.Timestamp,
    p_schedule: pd.DataFrame,
    pi_windows: int = 3,
    level: list[int] = [95],
) -> pd.DataFrame:
    """
    Backtesting OOS maison (cutoffs every step_size) avec p qui change selon le schedule.
    Utilise ConformalIntervals pour PI.
    """
    ts = ts.sort_values(["unique_id", "ds"]).reset_index(drop=True)
    ds_sorted = ts["ds"].sort_values().reset_index(drop=True)

    # cutoffs OOS: derniers points < ds - h
    # on part du dernier point avant test_start_ts comme ancre
    mask = ds_sorted < test_start_ts
    if not mask.any():
        raise ValueError("TEST_START trop tôt: aucune date avant.")
    first_cutoff = ds_sorted[mask].iloc[-1]

    # indices cutoffs (dans la timeline complète)
    first_idx = int(ds_sorted[ds_sorted == first_cutoff].index[0])
    last_cutoff_idx = len(ds_sorted) - 1 - h
    cutoff_indices = list(range(first_idx, last_cutoff_idx + 1, step_size))

    # helper: p* en vigueur à une date de cutoff = dernier reopt_date <= cutoff
    sched = p_schedule.copy()
    sched["reopt_date"] = pd.to_datetime(sched["reopt_date"], utc=True)

    out_rows = []
    ci = ConformalIntervals(h=h, n_windows=pi_windows)

    for cut_idx in cutoff_indices:
        cutoff_date = ds_sorted.iloc[cut_idx]

        # déterminer p courant
        elig = sched[sched["reopt_date"] <= cutoff_date]
        if len(elig) == 0:
            # si cutoff est avant 1983-01, pas de p* (mais en OOS tu seras >= 1990, donc OK)
            continue
        p_cur = int(elig.iloc[-1]["p_star"])

        train_df = ts[ts["ds"] <= cutoff_date].copy()

        sf_tmp = StatsForecast(models=[AutoRegressive(lags=p_cur)], freq=freq)
        fcst_df = sf_tmp.forecast(
            df=train_df,
            h=h,
            prediction_intervals=ci,
            level=level,
        )

        model_col = [c for c in fcst_df.columns if c.lower().startswith("autoregressive")][0]
        lo_col = [c for c in fcst_df.columns if c.lower().startswith("autoregressive") and c.lower().endswith(f"lo-{level[0]}")][0]
        hi_col = [c for c in fcst_df.columns if c.lower().startswith("autoregressive") and c.lower().endswith(f"hi-{level[0]}")][0]

        # vérité future
        y_true = ts[ts["ds"].isin(fcst_df["ds"])][["unique_id", "ds", "y"]]
        merged = fcst_df.merge(y_true, on=["unique_id", "ds"], how="inner")
        if len(merged) == 0:
            continue

        merged = merged.assign(cutoff=cutoff_date, p_used=p_cur)

        out_rows.append(
            merged[["unique_id", "ds", "cutoff", "p_used", "y", model_col, lo_col, hi_col]]
        )

    if len(out_rows) == 0:
        raise ValueError("Aucun résultat de backtest généré.")

    bkt = pd.concat(out_rows, ignore_index=True)

    # garder uniquement test >= TEST_START
    bkt = bkt[bkt["ds"] >= test_start_ts].reset_index(drop=True)

    return bkt

# Run 

In [64]:
# ----------------------------
# Config
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"

FEATURE_REFS = ["stationary_value:value"]  # y uniquement

H = 12
STEP_SIZE = 12
TEST_START = "1990-01-01"
PARTITIONS = 35
PI_WINDOWS = 3
LEVELS = [95]

AR_LAGS = 12 # Ajout

P_SELECTION_START = pd.Timestamp("1983-01-01", tz="UTC") # Ajout

p_grid = list(range(1, 13))  # ex: 1..12

REOPT_EVERY_MONTHS = 36  # <= ton besoin

In [65]:
# ----------------------------
# Rolling p* every 36 months (start 1983-01), expanding train from 1959
# ----------------------------

def mae_rolling_house(
    ts: pd.DataFrame,
    p: int,
    *,
    h: int,
    step_size: int,
    test_start_ts: pd.Timestamp,   # "fin" des données autorisées (strictement <)
    freq: str,
    p_selection_start: pd.Timestamp
) -> float:
    # TRAIN ONLY avant test_start_ts (évite leakage)
    ts_train = ts[ts["ds"] < test_start_ts].copy()
    ts_train = ts_train.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    ds_sorted = ts_train["ds"].sort_values().reset_index(drop=True)
    N = len(ds_sorted)

    last_cutoff_idx = N - 1 - h
    if last_cutoff_idx <= p:
        raise ValueError("Pas assez de points dans TRAIN pour évaluer ce p avec h donné.")

    first_cutoff_idx = ds_sorted[ds_sorted >= p_selection_start].index.min()
    if pd.isna(first_cutoff_idx):
        raise ValueError("p_selection_start est après la fin de la période TRAIN.")
    first_cutoff_idx = int(first_cutoff_idx)

    cutoff_indices = list(range(first_cutoff_idx, last_cutoff_idx + 1, step_size))
    if len(cutoff_indices) == 0:
        raise ValueError("Aucun cutoff généré (vérifie p_selection_start, h, step_size).")

    maes = []
    for cut_idx in cutoff_indices:
        cutoff_date = ds_sorted.iloc[cut_idx]
        train_df = ts_train[ts_train["ds"] <= cutoff_date].copy()  # expanding: 1959 -> cutoff

        sf_tmp = StatsForecast(models=[AutoRegressive(lags=p)], freq=freq)
        fcst_df = sf_tmp.forecast(df=train_df, h=h)

        model_col = [c for c in fcst_df.columns if c.lower().startswith("autoregressive")][0]

        y_true = ts_train[ts_train["ds"].isin(fcst_df["ds"])][["unique_id", "ds", "y"]]
        merged = fcst_df.merge(y_true, on=["unique_id", "ds"], how="inner")
        if len(merged) == 0:
            continue

        maes.append(float(np.mean(np.abs(merged["y"].values - merged[model_col].values))))

    if len(maes) == 0:
        raise ValueError("Impossible de calculer MAE (merge vide partout).")

    return float(np.mean(maes))


def compute_pstar_schedule(
    ts: pd.DataFrame,
    *,
    p_grid: list[int],
    freq: str,
    h: int,
    step_size: int,
    test_start_ts: pd.Timestamp,
    p_selection_start: pd.Timestamp,
    reopt_every_months: int
) -> pd.DataFrame:
    """
    Calcule p* à des dates de re-optimisation espacées de reopt_every_months,
    en commençant à p_selection_start (1983-01).
    Chaque p* est appris en utilisant uniquement les données < reopt_date (train),
    et la CV MAE rolling commence à 1983-01.
    """
    # Grille des dates de re-opt: 1983-01, 1986-01, 1989-01, ..., < TEST_START
    ts["ds"] = pd.to_datetime(ts["ds"], utc=True)

    uid = "UNRATE"
    last_date = ts.loc[ts["unique_id"] == uid, "ds"].max()
    last_cutoff = last_date - pd.DateOffset(months=h)

    reopt_dates = pd.date_range(
        start=p_selection_start + pd.DateOffset(months=reopt_every_months),
        end=last_cutoff,
        freq=f"{reopt_every_months}MS")

    rows = []
    for reopt_date in reopt_dates:
        mae_by_p = {
            p: mae_rolling_house(
                ts, p,
                h=h,
                step_size=step_size,
                test_start_ts=reopt_date,   # IMPORTANT: on n'utilise que le passé (< reopt_date)
                freq=freq,
                p_selection_start=p_selection_start
            )
            for p in p_grid
        }
        p_star = min(mae_by_p, key=mae_by_p.get)
        rows.append({
            "reopt_date": reopt_date.tz_convert("UTC") if reopt_date.tzinfo else reopt_date.tz_localize("UTC"),
            "p_star": int(p_star),
            "mae": float(mae_by_p[p_star])
        })

    return pd.DataFrame(rows).sort_values("reopt_date").reset_index(drop=True)

In [66]:
# ----------------------------
# 1) entity_df
# ----------------------------
dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

In [67]:
# ----------------------------
# 2) Feast -> ts (format StatsForecast)
# ----------------------------
ts_raw = load_features_from_feast(entity_df=entity_df, feature_refs=FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


In [68]:
# ----------------------------
# (RUN) 1) construire le schedule p* tous les 36 mois (1983 -> < 1990)
# ----------------------------
test_start_ts = pd.Timestamp(TEST_START, tz="UTC")

p_schedule = compute_pstar_schedule(
    ts,
    p_grid=p_grid,
    freq=FREQ,
    h=H,
    step_size=STEP_SIZE,
    test_start_ts=test_start_ts,
    p_selection_start=P_SELECTION_START,
    reopt_every_months=REOPT_EVERY_MONTHS,
)

print(p_schedule)

# ----------------------------
# (RUN) 2) backtesting OOS avec p dynamique
# ----------------------------
bkt_df_dyn = backtest_dynamic_p_oos(
    ts,
    freq=FREQ,
    h=H,
    step_size=STEP_SIZE,
    test_start_ts=test_start_ts,
    p_schedule=p_schedule,
    pi_windows=PI_WINDOWS,
    level=LEVELS,
)

bkt_df_dyn.head()

                  reopt_date  p_star       mae
0  1986-01-01 00:00:00+00:00      12  0.551024
1  1989-01-01 00:00:00+00:00       6  0.644701
2  1992-01-01 00:00:00+00:00       3  0.521484
3  1995-01-01 00:00:00+00:00       6  0.485479
4  1998-01-01 00:00:00+00:00       3  0.456492
5  2001-01-01 00:00:00+00:00       3  0.404259
6  2004-01-01 00:00:00+00:00       4  0.391902
7  2007-01-01 00:00:00+00:00       4  0.382031
8  2010-01-01 00:00:00+00:00       4  0.394063
9  2013-01-01 00:00:00+00:00       5  0.417801
10 2016-01-01 00:00:00+00:00       4  0.435952
11 2019-01-01 00:00:00+00:00       4  0.439293
12 2022-01-01 00:00:00+00:00       4  0.552780


,unique_id,ds,cutoff,p_used,y,AutoRegressive,AutoRegressive-lo-95,AutoRegressive-hi-95
0,UNRATE,1990-01-01 00:00:00+00:00,1989-12-01 00:00:00+00:00,6,0.0,0.186394,-0.024703,0.397490
1,UNRATE,1990-02-01 00:00:00+00:00,1989-12-01 00:00:00+00:00,6,0.1,0.224738,0.016575,0.432901
2,UNRATE,1990-03-01 00:00:00+00:00,1989-12-01 00:00:00+00:00,6,0.2,0.264480,-0.157540,0.686500
3,UNRATE,1990-04-01 00:00:00+00:00,1989-12-01 00:00:00+00:00,6,0.2,0.282086,-0.098931,0.663103
4,UNRATE,1990-05-01 00:00:00+00:00,1989-12-01 00:00:00+00:00,6,0.2,0.290851,-0.185308,0.767010


In [69]:
# ----------------------------
# 3) StatsForecast
# ----------------------------
sf = StatsForecast(
    models=[AutoRegressive(lags=AR_LAGS)],
    freq=FREQ,
)

In [70]:
# ----------------------------
# 4) Cross-validation + Conformal intervals
# ----------------------------
ci = ConformalIntervals(h=H, n_windows=PI_WINDOWS)

test_start_ts = pd.Timestamp(TEST_START, tz="UTC")
ds_sorted = ts["ds"].sort_values().reset_index(drop=True)

mask = ds_sorted < test_start_ts
if not mask.any():
    raise ValueError(f"TEST_START={TEST_START} est trop tôt (aucune date avant dans ts['ds']).")

cutoff_date = ds_sorted[mask].iloc[-1]
c = int(ds_sorted[ds_sorted == cutoff_date].index[0])
N = len(ds_sorted)

n_windows = int((((N - 1 - H) - c) // STEP_SIZE) + 1)
if n_windows <= 0:
    raise ValueError(f"TEST_START={TEST_START} est trop tard pour h={H} et step_size={STEP_SIZE}.")

bkt_df = sf.cross_validation(
    df=ts,
    h=H,
    step_size=STEP_SIZE,
    n_windows=n_windows,
    prediction_intervals=ci,
    level=LEVELS,
)

# Garder uniquement la partie test à partir de TEST_START
bkt_df = bkt_df[bkt_df["ds"] >= test_start_ts].reset_index(drop=True)

In [71]:
# ----------------------------
# 5) Reusable output table (FINAL)
# ----------------------------
# NB: le nom de la colonne modèle dépend du "alias" StatsForecast.
# Par défaut c'est souvent "AutoRegressive" (ou un nom proche).
model_col = [c for c in bkt_df.columns if c.lower().startswith("autoregressive")][0]

lo_col = [c for c in bkt_df.columns
          if c.lower().startswith("autoregressive") and c.lower().endswith("lo-95")][0]
hi_col = [c for c in bkt_df.columns
          if c.lower().startswith("autoregressive") and c.lower().endswith("hi-95")][0]

df_ar_forecasts = (
    bkt_df[["unique_id", "ds", "cutoff", "y", model_col, lo_col, hi_col]]
    .rename(columns={
        "unique_id": "series_id",
        "ds": "date",
        "y": "y_obs",
        model_col: "y_hat_ar",
        lo_col: "y_hat_ar_lo_95",
        hi_col: "y_hat_ar_hi_95",
    })
    # dates propres (sans +00:00), pratique pour plots / export
    .assign(
        date=lambda d: pd.to_datetime(d["date"]).dt.tz_localize(None),
        cutoff=lambda d: pd.to_datetime(d["cutoff"]).dt.tz_localize(None),
    )
    .sort_values(["series_id", "date"])
    .reset_index(drop=True)
)

df_ar_forecasts

,series_id,date,cutoff,y_obs,y_hat_ar,y_hat_ar_lo_95,y_hat_ar_hi_95
0,UNRATE,1990-10-01,1990-09-01,0.6,0.643004,0.455619,0.830389
1,UNRATE,1990-11-01,1990-09-01,0.8,0.701426,0.488123,0.914728
2,UNRATE,1990-12-01,1990-09-01,0.9,0.700045,0.499437,0.900653
3,UNRATE,1991-01-01,1990-09-01,1.0,0.725013,0.548912,0.901113
4,UNRATE,1991-02-01,1990-09-01,1.3,0.689714,0.244503,1.134924
...,...,...,...,...,...,...,...
415,UNRATE,2025-05-01,2024-09-01,0.2,-0.029221,-0.705199,0.646756
416,UNRATE,2025-06-01,2024-09-01,0.0,-0.083424,-1.261649,1.094802
417,UNRATE,2025-07-01,2024-09-01,0.0,-0.155028,-1.427002,1.116947
418,UNRATE,2025-08-01,2024-09-01,0.1,-0.193472,-1.142118,0.755173


# Graphique

In [72]:
df_obs = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })
    [["unique_id", "ds", "y"]]
)

In [73]:
df_fcst = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_hat_ar": "AR",
        "y_hat_ar_lo_95": "AR-lo-95",
        "y_hat_ar_hi_95": "AR-hi-95",
    })
    [[
        "unique_id",
        "ds",
        "AR",
        "AR-lo-95",
        "AR-hi-95",
    ]]
)

In [74]:
from utilsforecast.plotting import plot_series

fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

# Rename legend entries
for trace in fig.data:
    if trace.name == "y":
        trace.name = "Unemployment rate (%)"
    elif trace.name == "AR":
        trace.name = "AutoRegressive (AR)"
    elif "level_95" in trace.name.lower():
        trace.name = "95% Prediction Interval"

fig.show()

## Résultat
Globalement, le modèle auto-régressif reste proche des observations en période de stabilité. C'est une bonne capacité à capter la dynamique du chômage.

Lors des ruptures structurelles (crise de 2008, Covid-19), la qualité des prévisions se dégrade et les intervalles de prédiction s’élargissent. Ce qui qui traduit une incertitude de plus en plus élevée.

Le modèle capte la direction des variations, mais sa fiabilité diminue en période de crise, sans masquer cette incertitude.

Cette étude constitue un **sanity check du système de prévision**. Elle valide le comportement attendu du modèle et la cohérence du pipeline. Une approche plus complexe est attendu. 

## Next
Essayons d'optimiser le paramètre "p" de AR pour confirmer notre analyse. 